# **Algoritmo 1: Construcción de las matrices A y M**

In [ ]:
import numpy as np

#ENTRADAS
L=float(input("Ingrese la longitud de la cuerda L en metros: ")) #longitud de la cuerda
T=float(input("Ingrese la tensión de la cuerda T en Newtons: ")) #tensión de la cuerda
N=int(input("Ingrese el número de nodos N: ")) #número de nodos
mu1=float(input("Ingrese la densidad lineal inicial mu1 en x=0 (kg/m): ")) #densidad lineal inicial
mu2=float(input("Ingrese la densidad lineal final mu2 en x=L (kg/m):")) #densidad lineal final

def mu(x):
  return mu1+((mu2-mu1)*x)/L #densidad lineal

dx=L/(N+1) #separación entre nodos

#POSICIÓN DE LOS NODOS INTERIORES
i_indices=np.arange(1, N+1)
x=i_indices*dx

A=np.zeros((N, N)) #matriz de ceros A
M=np.diag(mu(x)) #matriz diagonal de densidad lineal M

for i in range(1, N+1):
  idx= i-1 #las componentes de la matriz se enumeran desde cero
  A[idx, idx]=2*T/(dx**2) #diagonal principal de matriz A
  if i>1:
    A[idx, idx-1]=-T/(dx**2) #diagonal inferior de matriz A
  if i<N:
    A[idx, idx+1]=-T/(dx**2) #diagonal superior de matriz A

# Opción más corta usando f-strings
print(f"\n--- MATRIZ A ---\n{np.round(A, 4)}")
print(f"\n--- MATRIZ M ---\n{np.round(M, 4)}")




Ingrese la longitud de la cuerda L en metros: 1
Ingrese la tensión de la cuerda T en Newtons: 10
Ingrese el número de nodos N: 15
Ingrese la densidad lineal inicial mu1 en x=0 (kg/m): 5
Ingrese la densidad lineal final mu2 en x=L (kg/m):10

--- MATRIZ A ---
[[ 5120. -2560.     0.     0.     0.     0.     0.     0.     0.     0.
      0.     0.     0.     0.     0.]
 [-2560.  5120. -2560.     0.     0.     0.     0.     0.     0.     0.
      0.     0.     0.     0.     0.]
 [    0. -2560.  5120. -2560.     0.     0.     0.     0.     0.     0.
      0.     0.     0.     0.     0.]
 [    0.     0. -2560.  5120. -2560.     0.     0.     0.     0.     0.
      0.     0.     0.     0.     0.]
 [    0.     0.     0. -2560.  5120. -2560.     0.     0.     0.     0.
      0.     0.     0.     0.     0.]
 [    0.     0.     0.     0. -2560.  5120. -2560.     0.     0.     0.
      0.     0.     0.     0.     0.]
 [    0.     0.     0.     0.     0. -2560.  5120. -2560.     0.     0.
      0.  


# **Algoritmo 2: Transformación del problema generalizado**


In [ ]:
D=np.zeros((N, N))
for i in range(1, N+1):
  idx=i-1
  D[idx, idx]= 1/np.sqrt(M[idx, idx]) #matriz D directamente de la raíz de la matriz diagonal M

C=D@A@D #Producto matricial C = D * A * D

print(f"\n--- MATRIZ C ---\n{np.round(C, 4)}")


--- MATRIZ C ---
[[ 963.7647 -468.3055    0.        0.        0.        0.        0.
     0.        0.        0.        0.        0.        0.        0.
     0.    ]
 [-468.3055  910.2222 -442.9726    0.        0.        0.        0.
     0.        0.        0.        0.        0.        0.        0.
     0.    ]
 [   0.     -442.9726  862.3158 -420.2407    0.        0.        0.
     0.        0.        0.        0.        0.        0.        0.
     0.    ]
 [   0.        0.     -420.2407  819.2    -399.7287    0.        0.
     0.        0.        0.        0.        0.        0.        0.
     0.    ]
 [   0.        0.        0.     -399.7287  780.1905 -381.1263    0.
     0.        0.        0.        0.        0.        0.        0.
     0.    ]
 [   0.        0.        0.        0.     -381.1263  744.7273 -364.1788
     0.        0.        0.        0.        0.        0.        0.
     0.    ]
 [   0.        0.        0.        0.        0.     -364.1788  712.3478
  -348.6747 

# **Algoritmo 3: Método de Jacobi para una matriz NxN**

In [ ]:
from sys import set_int_max_str_digits
#Parámetros de convergencia
tol=1e-8 #tolerancia
max_iter=500 #máxima iteración

V=np.eye(N) #inicializar V como matriz identidad NxN

for k in range(max_iter): #bucle principal de Jacobi
  s=0.0 #registra la información del candidato a máximo valor absoluto
  p, q=0, 0 #guardan fila y columna del elemento máximo encontrado
  #buscar el mayor elemento fuera de la diagonal principal
  for i in range(N-1): #se detiene en la penúltima fila
    for j in range(i+1, N): #recorre las columnas empezando siempre a la derecha de la diagonal
      if abs(C[i, j])>s: #compara el valor absoluto del elemento actual
        s=abs(C[i, j]) #solo si encuentra un número más grande que s
        p, q=i, j #actualiza la ubicación
  if s<tol: #si cumple lacondición de tolerancia se detiene el código
     break

  #Calculamos el ángulo theta en radianes
  if C[p, p]==C[q, q]:
    theta=np.pi/4 if C[p, q]>0 else -np.pi/4 #para el caso conde el ángulo tiende a infinito
  else:
    theta=0.5*np.arctan2(2*C[p, q], C[p, p]-C[q, q])

  #Contrucción de la matriz de rotación Q
  Q=np.eye(N)
  sin_t=np.sin(theta)
  cos_t=np.cos(theta)

  Q[p, p]=cos_t
  Q[q, q]=cos_t
  Q[p, q]=-sin_t
  Q[q, p]= sin_t

  C=Q.T@C@Q
  V=V@Q
lambdas=np.diag(C) #vector de autovalores
Y=V #los vectores porpios son la columnas de la matriz V

print(f"\n--- MATRIZ V ---\n{np.round(V, 4)}")
print(f"\n--- MATRIZ C ---\n{np.round(C, 4)}")


--- MATRIZ V ---
[[ 3.983e-01  3.335e-01  2.910e-01  2.625e-01 -2.594e-01  2.754e-01
   2.861e-01  2.864e-01  2.766e-01  2.574e-01 -2.295e-01 -1.938e-01
  -1.515e-01  1.041e-01  5.310e-02]
 [-5.779e-01 -3.203e-01 -1.839e-01 -1.005e-01  4.330e-02  1.770e-02
   9.250e-02  1.719e-01  2.438e-01  2.970e-01 -3.227e-01 -3.155e-01
  -2.744e-01  2.026e-01  1.078e-01]
 [ 5.352e-01  1.150e-02 -1.626e-01 -2.246e-01  2.614e-01 -2.921e-01
  -2.820e-01 -2.145e-01 -9.480e-02  5.420e-02 -1.981e-01 -3.001e-01
  -3.318e-01  2.824e-01  1.622e-01]
 [-3.855e-01  3.226e-01  3.475e-01  2.560e-01 -1.573e-01  3.100e-02
  -1.311e-01 -2.729e-01 -3.272e-01 -2.565e-01  7.760e-02 -1.394e-01
  -3.002e-01  3.308e-01  2.142e-01]
 [ 2.321e-01 -4.918e-01 -2.120e-01  2.880e-02 -1.872e-01  2.982e-01
   2.942e-01  1.323e-01 -1.198e-01 -3.109e-01  3.080e-01  1.001e-01
  -1.794e-01  3.378e-01  2.614e-01]
 [-1.212e-01  4.790e-01 -9.770e-02 -2.959e-01  2.935e-01 -1.527e-01
   1.127e-01  3.200e-01  2.711e-01 -2.200e-02  3.024e-

# **Algoritmo 4: Recuperamos los modos de la cuerda**

In [ ]:
X_interiores= D@Y #recuperamos las amplitudes interiores sin tomar los extremos
X_norm= X_interiores / np.max(np.abs(X_interiores)) #hacer que el máximo elemento de cada modo sea 1 para graficar más fácil
X_completos=np.zeros((N+2, N))
X_completos[1:-1, :]=X_norm #amplitudes tomando los extremos
x_completo=np.linspace(0, L, N+2) #arreglo de posiciones completas
print(f"\n--- MODOS NORMALIZADOS (NODOS INTERIORES) ---\n{np.round(X_norm, 4)}")
print(f"\n--- MODO 1 COMPLETO (con fronteras en 0) ---\n{np.round(X_completos[:, 0], 4)}")


--- MODOS NORMALIZADOS (NODOS INTERIORES) ---
[[ 7.092e-01  5.938e-01  5.181e-01  4.673e-01 -4.619e-01  4.904e-01
   5.095e-01  5.100e-01  4.926e-01  4.584e-01 -4.087e-01 -3.451e-01
  -2.698e-01  1.853e-01  9.460e-02]
 [-1.000e+00 -5.544e-01 -3.182e-01 -1.739e-01  7.490e-02  3.060e-02
   1.601e-01  2.975e-01  4.218e-01  5.139e-01 -5.585e-01 -5.460e-01
  -4.748e-01  3.506e-01  1.866e-01]
 [ 9.015e-01  1.940e-02 -2.738e-01 -3.784e-01  4.403e-01 -4.920e-01
  -4.750e-01 -3.612e-01 -1.597e-01  9.130e-02 -3.336e-01 -5.054e-01
  -5.589e-01  4.756e-01  2.732e-01]
 [-6.329e-01  5.296e-01  5.705e-01  4.203e-01 -2.582e-01  5.090e-02
  -2.152e-01 -4.480e-01 -5.371e-01 -4.211e-01  1.274e-01 -2.288e-01
  -4.929e-01  5.430e-01  3.516e-01]
 [ 3.719e-01 -7.879e-01 -3.397e-01  4.610e-02 -2.999e-01  4.777e-01
   4.714e-01  2.119e-01 -1.919e-01 -4.980e-01  4.935e-01  1.603e-01
  -2.874e-01  5.412e-01  4.188e-01]
 [-1.898e-01  7.498e-01 -1.530e-01 -4.631e-01  4.594e-01 -2.390e-01
   1.764e-01  5.010e-01  

# **Algoritmo 5: Cálculo de las frecuencias naturales**

In [ ]:
indices_ordenados=np.argsort(lambdas) #organizamos de menor a mayor los valores propios
lambdas=lambdas[indices_ordenados]
Y=Y[:, indices_ordenados] #asociamos su vector propio con su valor propio
if 'X_interiores' in locals():
  X_interiores=X_interiores[: , indices_ordenados]
  X_completos=X_completos[:, indices_ordenados]

omega=np.sqrt(lambdas)
frecuencia_hz=omega/(2*np.pi)
print(f"\n--- VALORES PROPIOS ORDENADOS (lambda) ---\n{np.round(lambdas, 4)}")
print(f"\n--- FRECUENCIAS ANGULARES (omega [rad/s]) ---\n{np.round(omega, 4)}")
print(f"\n--- FRECUENCIAS NATURALES (f [Hz]) ---\n{np.round(frecuencia_hz, 4)}")



--- VALORES PROPIOS ORDENADOS (lambda) ---
[  13.054    52.2353  115.8404  201.3286  305.2779  423.4856  551.0873
  682.6667  812.3205  933.7311 1041.8582 1143.0683 1259.6763 1413.6231
 1643.2681]

--- FRECUENCIAS ANGULARES (omega [rad/s]) ---
[ 3.613   7.2274 10.7629 14.189  17.4722 20.5788 23.4752 26.1279 28.5012
 30.557  32.2778 33.8093 35.4919 37.5982 40.5372]

--- FRECUENCIAS NATURALES (f [Hz]) ---
[0.575  1.1503 1.713  2.2583 2.7808 3.2752 3.7362 4.1584 4.5361 4.8633
 5.1372 5.3809 5.6487 5.9839 6.4517]
